Modelo MLP

# Stack = Pytorch

1. Definir arquitetura

Arquitetura conceitual
Neuronicos, ativação
Camadas (entrada; ocultas; saidas = cada camada transforma representações)
    a - entrada: dados inseridos na cadamda de entrada da rede
    b - camadas ocultas: dados passados para a primeira camada oculta, na qual são feitas três operações
    c - propagação sequencial: saida de uma camada se torna a entrada de outra
    d - saida final: produz o resultado final da rede, pode  ser classificaçãou ou numerico
Camadas ocultas: mais camadas e neur^nios e mais capacidade de representação
Trade off: profundidade/complexidade vs overfitting;

Capacidade e generalização
- Arquitetura especializadas;
- Validação e early stopiing para controlar a generalização.

BackPropagation - otimizacao
    1.perda.
    2.graientes: regra da cadeia = computçaõ eficiente pelos grafos.
    3.otimizadores: sgd, momentum, adam, rmsprop.
    4.hiperparametros: taxa de aprendizado, tamanho do batch, numero de épocas.


2. Função de ativação
3. Loss function


# Pré treino e tunning
Refinamento vai orientar profundidade e largura


# Função de ativação
As funções vão estabilizar os gradientes que vão viabilizar as tecnicas de backpropagation:
    1.Sigmoide e tanh: boa interpretação, risco de vanishing gradientes.
    2.Relu: (max\90,z) simples e eficiente, variantes leakyrelu, elu, gelu
    3.softmax: normaliza logits em probabilidades (multiclasse)


Overfiting e regularização
fundamentos:
    1.Sinais: alto no treino, baixo no teste; curva de aprendizado divergente.
    2.Tecnicas: L2; Dropout; Early stopping, data augmentation.
    3.BatchNorm ajuda estabilidade; pode atuar como regularizador indireto.


Linear/logistica: rapida e interpretavel; limite em padroes complexos
Arvore/ensemble: forte em dados tabulares
Redes neurais: poderosas em dados riccos, exigem mais dados

In [ ]:
#Checklist para evitar overfitting: antipadrões
#1.Normalizar inputs, escolher ativação adequada

#2.Definir perda/metricas corretas (cross entropy, AUC-ROC, accuracy) e monitorar durante treino

#3.Usar early stopping e regularização: monitorar overfitting;

#4.Verificar calibração quando ncessario e robustez a ruido/outliers

In [ ]:
import pandas as pd
import numpy as np
import subprocess
import sys
import mlflow
import mlflow.sklearn
import mlflow.data
import matplotlib.pyplot as plt # Adicionado para o gráfico
import os # Adicionado para gerenciar arquivos
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [ ]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [20]:
# 1. Configurações de separação
test_size = 0.2
random_state = 42

# Separando variáveis alvo e features
X = df.drop(columns=['churn_value'])
y = df['churn_value']

# Split com estratificação (importante para Churn!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

# Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Aplicando transformação
X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

feature_names = preprocessador.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

display(X_train_df.head())

,cat__country_United States,cat__state_California,cat__city_Acampo,cat__city_Acton,cat__city_Adelanto,cat__city_Adin,cat__city_Agoura Hills,cat__city_Aguanga,cat__city_Ahwahnee,cat__city_Alameda,...,cat__churn_reason_Network reliability,cat__churn_reason_Poor expertise of online support,cat__churn_reason_Poor expertise of phone support,cat__churn_reason_Price too high,cat__churn_reason_Product dissatisfaction,cat__churn_reason_Service dissatisfaction,num__count,num__tenure_months,num__monthly_charges,num__total_charges
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.109845,-0.520304,-0.257010
1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.703546,0.340040,-0.499098
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.784885,-0.807639,-0.746072
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.256181,0.286892,-0.167211
4,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.272920,-0.674767,-0.986268


In [28]:
#Preparação dos dados para PyTorch
import torch
from torch.utils.data import DataLoader, TensorDataset

# Converter DataFrames para Tensores PyTorch
X_train_tensor = torch.tensor(X_train_df.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_df.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Implementando Batching (Requisito Etapa 2)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [29]:
#Definição de arquitetura MLP
import torch.nn as nn

class ChurnMLP(nn.Module):
    def __init__(self, input_size):
        super(ChurnMLP, self).__init__()
        # Arquitetura: Camadas Lineares (Dense Layers)
        self.layer1 = nn.Linear(input_size, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        
        # Função de Ativação: ReLU para camadas ocultas e Sigmoid para a saída
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# Instanciando o modelo
input_dim = X_train_df.shape[1]
model = ChurnMLP(input_dim)

In [30]:
#Loss Function e Otimizador
criterion = nn.BCELoss() # Loss Function para classificação binária
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Otimizador Adam

In [32]:
#Early Stopping e Regularização
import numpy as np
import torch.nn as nn
import torch.optim as optim

# Ensure criterion and optimizer are defined
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100
patience = 10
best_loss = np.inf
counter = 0

for epoch in range(epochs):
    model.train()
    train_losses = []
    
    for batch_X, batch_y in train_loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass (Otimização)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # --- Lógica de Early Stopping ---
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_tensor)
        val_loss = criterion(val_outputs, y_test_tensor)
    
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        # Salvar o melhor modelo (Requisito de Artefato)
        torch.save(model.state_dict(), '../models/best_mlp_model.pt')
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping na época {epoch}")
            break

Early stopping na época 10


In [37]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
import mlflow
import mlflow.pytorch # Importação explícita para evitar erros de atributo
import torch

# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")


with mlflow.start_run(run_name="MLP_pytorch_final"):
    # 1. Log de Parâmetros (Requisito Etapa 2) 
    mlflow.log_params({
        "architecture": "64-32-1",
        "optimizer": "Adam",
        "early_stopping_patience": 10,
        "batch_size": 32
    })

    csv_path = "data/processed/dataset_tratado.parquet"
    dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")
    mlflow.log_input(dataset, context="training")
    
    # 2. Carregando o estado salvo pelo Early Stopping 
    model.load_state_dict(torch.load('../models/best_mlp_model.pt'))
    model.eval()
    
    with torch.no_grad():
        # Predições para o teste
        y_probs = model(X_test_tensor).numpy()
        y_pred = (y_probs > 0.5).astype(int)
        
    # 3. Registrar as ≥ 4 métricas exigidas (Etapa 2) 
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_probs)
    }
    mlflow.log_metrics(metrics)
    
    # 4. Log do modelo como artefato PyTorch (Etapa 2) [cite: 50, 52]
    mlflow.pytorch.log_model(model, "mlp_model")
    
    print("Sucesso: Experimento MLP registrado no MLflow.")

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Sucesso: Experimento MLP registrado no MLflow.
